# p-Laplace 2D — FEniCS Benchmark

This notebook runs the FEniCS (DOLFINx) p-Laplace solvers and displays benchmark results.

**Prerequisites**: Run this notebook inside the devcontainer (or any environment where DOLFINx >= 0.10 is available).

The solvers are in `pLaplace2D_fenics/`:
- `solve_pLaplace_snes_newton.py` — SNES Newton (recommended)
- `solve_pLaplace_custom_jaxversion.py` — Custom Newton matching JAX algorithm (line search on [−0.5, 2], tighter tolerances)

Meshes are in `mesh_data/pLaplace/` (XDMF files, generated from HDF5 source via `pLaplace2D_fenics/export_pLaplace_meshes.py`).

In [ ]:
import subprocess, json, os, sys
import pandas as pd
from IPython.display import Markdown, display

print(f"Working directory: {os.getcwd()}")

## Helper: run a solver and collect JSON results

In [ ]:
def run_solver(script, nprocs=1, levels=None, extra_args=None):
    """Run a solver script (optionally with mpirun) and return parsed JSON results."""
    if levels is None:
        levels = [5, 6, 7, 8, 9]
    
    tmp_json = "/tmp/_benchmark_result.json"
    cmd = []
    if nprocs > 1:
        cmd = ["mpirun", "-n", str(nprocs)]
    cmd += ["python3", script, "--levels"] + [str(l) for l in levels]
    cmd += ["--json", tmp_json]
    if extra_args:
        cmd += extra_args
    
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f"STDERR: {result.stderr}", file=sys.stderr)
        return None
    
    with open(tmp_json) as f:
        data = json.load(f)
    os.remove(tmp_json)
    return data

## 1. SNES Newton — Serial

In [ ]:
snes_serial = run_solver("pLaplace2D_fenics/solve_pLaplace_snes_newton.py", nprocs=1)
if snes_serial:
    df = pd.DataFrame(snes_serial["results"])
    df = df.rename(columns={"mesh_level": "mesh_lvl", "total_dofs": "dofs"})
    display(df[["mesh_lvl", "dofs", "time", "iters", "energy"]].to_markdown(index=False))

## 2. SNES Newton — Parallel (4 processes)

In [ ]:
snes_4proc = run_solver("pLaplace2D_fenics/solve_pLaplace_snes_newton.py", nprocs=4)
if snes_4proc:
    df = pd.DataFrame(snes_4proc["results"])
    df = df.rename(columns={"mesh_level": "mesh_lvl", "total_dofs": "dofs"})
    display(df[["mesh_lvl", "dofs", "time", "iters", "energy"]].to_markdown(index=False))

## 3. Custom Newton (JAX algorithm) — Serial

Same Newton algorithm as the JAX solver (`tools/minimizers.py`): golden-section line search on [−0.5, 2] with tol=1e-3, gradient L2 stopping at 1e-3, energy change stopping at 1e-5, and CG + HYPRE AMG with rtol=1e-3.

In [ ]:
jaxver_serial = run_solver("pLaplace2D_fenics/solve_pLaplace_custom_jaxversion.py", nprocs=1, extra_args=["--quiet"])
if jaxver_serial:
    df = pd.DataFrame(jaxver_serial["results"])
    df = df.rename(columns={"mesh_level": "mesh_lvl", "total_dofs": "dofs"})
    display(df[["mesh_lvl", "dofs", "time", "iters", "energy"]].to_markdown(index=False))

## 4. Comparison Table

Combine all results into a single comparison table.

In [ ]:
def results_to_df(data, label):
    if data is None:
        return pd.DataFrame()
    df = pd.DataFrame(data["results"])
    df = df.rename(columns={"mesh_level": "mesh_lvl", "total_dofs": "dofs"})
    df["solver"] = label
    return df

frames = []
for data, label in [
    (snes_serial, "SNES serial"),
    (snes_4proc, "SNES 4-proc"),
    (jaxver_serial, "Custom serial"),
]:
    frames.append(results_to_df(data, label))

if frames:
    all_df = pd.concat(frames, ignore_index=True)
    pivot = all_df.pivot_table(index=["mesh_lvl", "dofs"], columns="solver", values=["time", "iters"])
    pivot.columns = [f"{col[1]} {col[0]}" for col in pivot.columns]
    display(Markdown(pivot.to_markdown()))

## 5. Batch Benchmark (multiple repetitions)

For proper benchmarking with median times, use `results/run_experiments.py`:

```bash
python3 results/run_experiments.py --nprocs 1 4 8 --levels 5 6 7 8 9 --repeat 3 --tag my_run
```

Then generate LaTeX/Markdown tables:

```bash
python3 results/generate_latex_tables.py results/<experiment_id>/ --markdown
```